In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from keras import callbacks, layers, models

2025-10-01 14:59:23.655447: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-01 14:59:23.733467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-01 14:59:25.450515: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
data = pd.read_csv('/home/onyxia/work/formation_cepe/Bloc3/data/ozone_complet.csv',
                   header    = 0,

                   sep       = ';',
                   decimal   = ',')

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1464 entries, 0 to 1463
Data columns (total 24 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    1464 non-null   int64  
 1   maxO3   1391 non-null   float64
 2   T6      1463 non-null   float64
 3   T9      1463 non-null   float64
 4   T12     1463 non-null   float64
 5   T15     1463 non-null   float64
 6   T18     1463 non-null   float64
 7   Ne6     1462 non-null   float64
 8   Ne9     1462 non-null   float64
 9   Ne12    1462 non-null   float64
 10  Ne15    1462 non-null   float64
 11  Ne18    1459 non-null   float64
 12  Vdir6   1463 non-null   float64
 13  Vvit6   1463 non-null   float64
 14  Vdir9   1463 non-null   float64
 15  Vvit9   1463 non-null   float64
 16  Vdir12  1463 non-null   float64
 17  Vvit12  1463 non-null   float64
 18  Vdir15  1463 non-null   float64
 19  Vvit15  1463 non-null   float64
 20  Vdir18  1463 non-null   float64
 21  Vvit18  1463 non-null   float64
 22  

In [36]:
data=data.dropna()

In [37]:
target = 'maxO3'

y = data[target]
X = data.drop(target, axis=1)

In [38]:
test_portion  = 1/5
valid_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)

print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

Dimensions de X_train : (873, 23)
Dimensions de X_valid : (219, 23)
Dimensions de X_test  : (274, 23)
Dimensions de y_train : (873,)
Dimensions de y_valid : (219,)
Dimensions de y_test  : (274,)


In [39]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_valid_norm = scaler.transform(X_valid)
X_test_norm  = scaler.transform(X_test)

In [40]:
dim_inputs  = (X_train_norm.shape[1],)
dim_outputs = 1

n_units_hl1 = 50
n_units_hl2 = 30

dropout_hl1 = 0.2
dropout_hl2 = 0.2

model = models.Sequential(name='DNN')

model.add(layers.Input(shape=dim_inputs, name='Inputs'))

model.add(layers.Dense(units=n_units_hl1, activation='relu', name='Hidden_layer_1'))
model.add(layers.Dropout(rate=dropout_hl1, name='Dropout_Hidden_layer_1'))

model.add(layers.Dense(units=n_units_hl2, activation='relu', name='Hidden_layer_2'))
model.add(layers.Dropout(rate=dropout_hl2, name='Dropout_Hidden_layer_2'))

model.add(layers.Dense(units=dim_outputs, activation='linear', name='Output_layer'))

model.summary()

Model: "DNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_layer_1 (Dense)          │ (None, 50)             │         1,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Hidden_layer_1          │ (None, 50)             │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_layer_2 (Dense)          │ (None, 30)             │         1,530 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Hidden_layer_2          │ (None, 30)             │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_layer (Dense)            │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,761 (10.79 KB)

 Trainable params: 2,761 (10.79 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer = 'adam',
              loss      = 'MSE',
              metrics   = ['MSE'])

callback = callbacks.EarlyStopping(monitor              = 'val_loss',
                                   mode                 = 'min',
                                   patience             = 20,
                                   restore_best_weights = True)

In [42]:
hist = model.fit(X_train_norm,
                 y_train,
                 batch_size      = 500,
                 epochs          = 200,
                 validation_data = (X_valid_norm, y_valid),
                 callbacks       = [callback],
                 verbose         = 1)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 382ms/step - MSE: 7748.4316 - loss: 7748.4316 - val_MSE: 7679.5605 - val_loss: 7679.5605
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 286ms/step - MSE: 7717.6860 - loss: 7717.6860 - val_MSE: 7653.5605 - val_loss: 7653.5605
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 348ms/step - MSE: 7695.0161 - loss: 7695.0161 - val_MSE: 7628.2051 - val_loss: 7628.2051
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 319ms/step - MSE: 7674.4512 - loss: 7674.4512 - val_MSE: 7603.3369 - val_loss: 7603.3369
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 295ms/step - MSE: 7645.9438 - loss: 7645.9438 - val_MSE: 7578.8477 - val_loss: 7578.8477
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 302ms/step - MSE: 7625.3105 - loss: 7625.3105 - val_MSE: 7554.4497 - val_loss: 7554.4497
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step - MSE: 7602.1201 - loss: 7602.1201 - val_MSE: 7530.0698 - val_loss: 7530.0698
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - MSE: 7580.2559 - loss: 7580.2559 - val_MSE: 